In [0]:
%run ../00-common/01-env-variables

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table  = f"{catalog_name}.{silver_schema}.results"

In [0]:
results_df = spark.read.table(bronze_table)
display(results_df)

In [0]:
renamed_results_df = (
    results_df.withColumnsRenamed({
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "raceName": "race_name",
        "positionText" :"finish_position_text",
        "date" : "reac_date",
        "grid" : "grid_position",
        "laps" : "completed_laps",
        "number" : "car_number",
        "position" : "finish_positiom"
    })
)

In [0]:
from pyspark.sql import functions as F
non_null_results_df = (
    renamed_results_df.filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull() )
)

In [0]:
distinct_results_df = non_null_results_df.dropDuplicates(['constructor_id' , 'driver_id' , 'season' , 'round'])


In [0]:
final_results_df = (
    distinct_results_df.drop('url')
    .withColumn('race_name' , F.initcap(F.col('race_name')))
)

In [0]:
final_results_df.write.mode('overwrite').saveAsTable(silver_table)